Note: 

Both SwapRateMCSimulator and SwapRatePCAEstimator require historical swap rate time series data in a specific format.
If no explicit file path is provided to the constructor, a default CSV file will be used instead.

The default dataset is located in the "curve_data" folder of the project (see "eur_swap_ts.csv" for the expected format).

In the examples below, a file path is explicitly provided, although it points to the same data as the default.

This notebook demonstrates how to:
- run full calculation batch and prepare data (parquet files) for streamlit app/dashboard from the alm_ir_hedge_app project.

In [6]:
from __future__ import annotations
import os 
from pathlib import Path

# It is assumed that the notebook is executed from within the project - either root or sub folder. 
CWD_PATH = Path.cwd()
SWAP_TIME_SERIES_CSV_PATH = ""

for parent in CWD_PATH.parents:
 if (parent / "src").exists():
     SWAP_TIME_SERIES_CSV_PATH = parent / "curve_data" / "eur_swap_ts.csv"
     break       
 else:
     SWAP_TIME_SERIES_CSV_PATH = CWD_PATH / "curve_data" / "eur_swap_ts.csv"

print(CWD_PATH)
print(SWAP_TIME_SERIES_CSV_PATH)   

C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\example_notebooks
C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\curve_data\eur_swap_ts.csv


In [7]:
from datetime import datetime
from data_pipeline.data_pipeline import create_calc_batch, create_parquet

In [8]:
# parameters
analysis_date = datetime(2026,3,31)

analysis_date_swap_rates = { 
 '2Y' : 2.948 , '3Y' : 2.934, '5Y' : 2.983, '7Y' : 3.036, '10Y': 3.110, '12Y': 3.195, 
 '15Y': 3.2439, '20Y': 3.246, '25Y': 3.186, '30Y': 3.115 }

liabilities_cashflow_schedule = { 
 '1Y': 24071664,  '2Y': 24931367,  '3Y': 26077636,  '4Y': 27510474,   
 '5Y': 29229878,  '6Y': 31235850,  '7Y': 33528390,  '8Y': 36107497, 
 '9Y': 38400036, '10Y': 40692576, '11Y': 39546306, '12Y': 41838845,
 '13Y': 44417952, '14Y': 47283627, '15Y': 50435869, '16Y': 53588110, 
 '17Y': 56167217, '18Y': 57886622, '19Y': 58746324, '20Y': 59032892,
 '21Y': 58459757, '22Y': 59032892, '23Y': 58459757, '24Y': 57313487,
 '25Y': 55594083, '26Y': 53301543, '27Y': 50435869, '28Y': 46997059, 
 '29Y': 42985115, '30Y': 38973171 }

In [9]:
# create batch - run all calculations - takes approx. 15-20 mins (with 1000 MC scenarios)
# calculations are saved as dict of dataframes. 
streamlit_data_dict = ( 
 create_calc_batch(
  analysis_date = analysis_date,  
  analysis_date_swap_rates = analysis_date_swap_rates,  
  liabilities_cashflow_schedule = liabilities_cashflow_schedule, 
  mc_n_scenarios = 1000, 
  mc_t_scale_factor = 30, 
  mc_rng_seed = 1, 
  swap_ts_csv_path = SWAP_TIME_SERIES_CSV_PATH
 ) 
) 

SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 4.554749889445546e-16, `time`: 0.1032s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0042s
Starting MC simulation of curve scenarios: 2026-05-05 20:58:30.341629


C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, spline end: 2056-04-05, curve id: 'eurswap'
This often occurs when a curve is constructed with a final node date that aligns with the maturity of an instrument with a payment lag.
In the case that the instrument has a payment lag (e.g. a SOFR swap or ESTR swap or bond terminating on a non-business day) then a cashflow will occur after the maturity of the instrument.
The solution is to ensure that the final node date of the curve is changed to be beyond that expected payment date.
  warnings.warn(


SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 3.13574640726366e-16, `time`: 0.0978s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 6.50531716466949e-16, `time`: 0.0912s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 5.137182868220651e-16, `time`: 0.0981s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 1.6703745712172888e-16, `time`: 0.0905s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 9.661113896766674e-16, `time`: 0.0950s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 3.4509528820842696e-16, `time`: 0.0908s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 7.586365078553314e-16, `time`: 0.0952s
SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 3.474079273925337e-16, `time`: 0.0898s
SUCCESS: `func_tol` reached after 7 iterations (levenber

C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, spline end: 2056-04-05, curve id: 'eurswap'
This often occurs when a curve is constructed with a final node date that aligns with the maturity of an instrument with a payment lag.
In the case that the instrument has a payment lag (e.g. a SOFR swap or ESTR swap or bond terminating on a non-business day) then a cashflow will occur after the maturity of the instrument.
The solution is to ensure that the final node date of the curve is changed to be beyond that expected payment date.
  warnings.warn(


SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0046s


...

SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0044s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0038s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0040s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0038s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0040s


C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, spline end: 2056-04-05, curve id: 'eurswap'
This often occurs when a curve is constructed with a final node date that aligns with the maturity of an instrument with a payment lag.
In the case that the instrument has a payment lag (e.g. a SOFR swap or ESTR swap or bond terminating on a non-business day) then a cashflow will occur after the maturity of the instrument.
The solution is to ensure that the final node date of the curve is changed to be beyond that expected payment date.
  warnings.warn(
C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, spline e

SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0039s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0039s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0042s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0039s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0038s
Starting MC simulation of PnL for Liability portfolios: 2026-05-05 21:01:07.258474
...ended at: 2026-05-05 21:07:12.414606


C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, spline end: 2056-04-05, curve id: 'eurswap'
This often occurs when a curve is constructed with a final node date that aligns with the maturity of an instrument with a payment lag.
In the case that the instrument has a payment lag (e.g. a SOFR swap or ESTR swap or bond terminating on a non-business day) then a cashflow will occur after the maturity of the instrument.
The solution is to ensure that the final node date of the curve is changed to be beyond that expected payment date.
  warnings.warn(


SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0039s


In [11]:
# save the dfs as parquet files. Files are placed in the alm_ir_hedging_prototype/streamlit_app_data folder. 
# from there, they can be copied and moved to the app_data of the streamlit app project alm_ir_hedging_app, to run the dashboard.
create_parquet(batch_df_dict = streamlit_data_dict) 